### PyAnsys Automation 

Using PyAnysy Fluent and geometry I will preform CFD runs to first intially gather data to train a ML model to predict outputs from a specifc generated geometry or train one of thei ML models which may be more efficeint and preform optimizations with my own optmization function


In [ ]:
import ansys.fluent.core as pyfluent
import os
import pandas as pd
from ansys.fluent.core.solver.function import reduction

#mesh cases
case_folder="cases" #like name  of folder with cfd files (like pre setup)
results_file='cfd_results.csv' #name of file putting reuslts in 

case_files = [ 
    "ME193EFINALV2.cas.h5",
    "ME193EFINALV3.cas.h5",
    "ME193EFINALV4.cas.h5",
    "ME193EFINALV5.cas.h5",
    "ME193EFINALV6.cas.h5",
    "ME193EFINALV7.cas.h5"
]

iterations=50 #change later for real CFD runs just testing

#launching fluent and setting the mode to solver and doing double presc and using 4 cores 
solver = pyfluent.launch_fluent(
    mode="solver",
    precision="double",
    processor_count=1,   # try 2 first for ANSYS Student
    ui_mode="gui",       # helps you see the real Fluent error
    start_timeout=120
)

results= [ ]

for case_name in case_files:
    case_path=os.path.join(case_folder,case_name) #ig its like os operating system takes path of current project opens folder 

    print(f" cfd run for {case_name}")

    solver.tui.file.read_case(case_path) #reading mesh filter

    #intialize
    solver.tui.solve.initialize.initialize_flow()

    #run solver
    solver.tui.solve.iterate(iterations)

    ### Now save results -> not surface int bc of typ error so reduce analgo to return one number rep
    inlet_p = reduction.area_average(
    expression="StaticPressure",
    locations=["inlet"],
    ctxt=solver
    )

    outlet_p = reduction.area_average(
        expression="StaticPressure",
        locations=["outlet"],
        ctxt=solver
    )

    inlet_T = reduction.area_average(
        expression="StaticTemperature",
        locations=["inlet"],
        ctxt=solver
    )

    outlet_T = reduction.area_average(
        expression="StaticTemperature",
        locations=["outlet"],
        ctxt=solver
    )

    delta_p= inlet_p-outlet_p
    delta_T= outlet_T-inlet_T

    print(case_name,delta_p,delta_T)

 # dictionary thing cool { } 
    results.append({
        "case": case_name,
        "inlet_pressure": inlet_p,
        "outlet_pressure": outlet_p,
        "delta_p": delta_p,
        "inlet_temp": inlet_T,
        "outlet_temp": outlet_T,
        "delta_T": delta_T,
    }) #essentially dicitionary gives a columnn name or category of data when pandas gets it alligns columns

    #save data

    data_name=case_name.replace(".cas.h5",'.dat.h5') #literally just gettting cfd file names and replacing the end with new file type
    solver.tui.file.write_data(os.path.join(case_folder,data_name))

#fix table and save 
df = pd.DataFrame(results)
df.to_csv(results_file, index=False) #now on csv

print("Done. Results saved to:", results_file) 

solver.exit()


 cfd run for ME193EFINALV2.cas.h5
Fast-loading "C:\PROGRA~1\ANSYSI~1\ANSYSS~1\v261\fluent\fluent26.1.0\\addons\afd\lib\hdfio.bin"
Done.

Reading from DESKTOP-H31RO31:"c:\Users\Admin\Desktop\Engineering Misc\ME 193E Projects\cases\ME193EFINALV2.cas.h5" in NODE0 mode ...
  Reading mesh ...
      666280 cells,     1 cell zone  ...
         666280 mixed cells,  zone id: 4
     1494101 faces,     5 face zones ...
          20125 mixed wall faces,  zone id: 1
        1416627 mixed interior faces,  zone id: 3
             73 triangular pressure-outlet faces,  zone id: 9
             64 triangular mass-flow-inlet faces,  zone id: 10
          57212 mixed wall faces,  zone id: 11
      211767 nodes,     1 node zone  ...
  Done.


Building...
     mesh
     materials,
     interface,
     domains,
	mixture
     zones,
	heating
	outlet
	interior-volume_volume
	wall-volume_volume
	inlet
	volume_volume
     parallel,
Done.
The following solver settings object method could also be used to execute th

pyfluent.launcher ERROR: Exception caught - RuntimeError: failed to connect to all addresses; last error: UNAVAILABLE: ipv4:127.0.0.1:61943: WSAGetOverlappedResult: Connection refused (No connection could be made because the target machine actively refused it.
 -- 10061)


LaunchFluentError: 
Fluent Launch command: "C:\Program Files\ANSYS Inc\ANSYS Student\v261\fluent\ntbin\win64\fluent.exe" 3ddp -py -gu -driver null -sifile=C:\Users\Admin\AppData\Local\Temp\serverinfo-mqcb34xn.txt -nm

: 

## MLP Training

Via feeding said CFD data into a MLP (better for tabulated data and predictions a Neural network with multiple layers ) it serves a surogate solver to use for optmization. Alongside my own trained MLP an Ansys ML model is also utilized to show the difference in the depth and trainning.


In [ ]:
import tensorflow as tf
import joblib 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

#pandas getting the data
df=pd.read_csv("cfd_results.csv") #df=direct file?

#edit later!!!!
input_columns = [
    "N_fin",
    "g_fin",
    "t_fin",
]


output_columns = [
    "delta_p",
    "delta_T"
]

X = df[input_columns].values
y = df[output_columns].values


# input outputs

input_col=[]

output_col=[] ## will get data later and put into there


X=[]

y=[]

X_train,X_test,y_train,y_test=train_test_split(
    X,y, test_size=0.2, random_state= 42
)

#Scale the data so each input signal doesnt dominate other

X_scaler= StandardScaler()
y_scaler= StandardScaler()

X_train_scaled=X_scaler.fit_transform(X_train)
X_test_scaled=X_scaler.transform(X_test)

y_train_scaled=y_scaler.fit_transform(y_train)
y_test_scaled=y_scaler.transform(y_test)

#building the mlp hoping layers can capture complexsity of relationship
S_CFD= tf.keras.Sequential([ 
    tf.keras.layers.Input(shape=(X_train_scaled.shape[1])),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(2),
])

#compile model or cfd aka choosing training settings/ learning settings

S_CFD.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

#trainning model

history= S_CFD.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=.2,
    epochs=100,
    batch_size=32
)

#evaluate model -> how accurate it is via a test with data

y_pred_scaled=S_CFD.predict(X_test_scaled)

y_pred=y_scaler.inverse_transform(y_pred_scaled)
y_test_real= y_scaler.inverse_transform(y_test_scaled)

# delta_p_mae ---> error evaluation do when get data

#save model

S_CFD.save("cold_plate_surogate_cfd.keras")
joblib.dump(X_scaler, "X_scaler.pkl")
joblib.dump(y_scaler, "y_scaler.pkl") #-> save trainning values scaling to use within actual model

# testing surgate
new_design = pd.DataFrame([{
    "N_fin": 0.002,
    "t_fin": 0.001,
    "g_fin": 0.12,
}])

new_design_scaled = X_scaler.transform(new_design[input_columns].values) #scaling inputs to the saved input scaling so effective prediciton

prediction_scaled = S_CFD.predict(new_design_scaled)
prediction = y_scaler.inverse_transform(prediction_scaled) #unscaling prediction from the scaled signals of midel

print("Predicted Delta P:", prediction[0, 0])
print("Predicted Delta T:", prediction[0, 1])

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

## Genetic Optimization Fitness Function

The fitness function used for optimization is:

$$
J(x) =
-1000\,\Delta T
-0.1\,\Delta P
+ P_{\text{gap}}
+ P_{\text{thickness}}
+ P_{\text{fin}}
$$

where the penalty terms are defined as:

$$
P_{\text{gap}} =
\begin{cases}
10000 & \text{if } g_{\text{fin}} < 0.2\ \text{mm} \\
0 & \text{otherwise}
\end{cases}
$$

$$
P_{\text{thickness}} =
\begin{cases}
10000 & \text{if } t_{\text{fin}} < 0.1\ \text{mm} \\
0 & \text{otherwise}
\end{cases}
$$

$$
P_{\text{fin}} =
\begin{cases}
10000 & \text{if } n_{\text{fin}} > n_{\text{fin,max}} \\
0 & \text{otherwise}
\end{cases}
$$

This objective minimizes temperature rise and pressure drop while penalizing infeasible geometries.

In [ ]:
#import the case data requirements
import random as rng

one= [1,1]
two= [1,5]
three= [3,1]
four= [2,5]
five= [5,2]
six= [1,10]
seven = [10,1]

iterations = 5

cases= [one,two,three,four,five,six,seven ] #each case 
#number of fins function defined in paremetresized geoemtry
Nfin = lambda gfin,tfin,L: round((L+gfin)/(tfin+gfin))*.5

def J_func(delT,delP,gfin,tfin):

    Pth = 10000 if tfin < .1 else 0 

    Pgap = 10000 if gfin  < 2 else 0 

    Pfin = 10000 if Nfin(gfin,tfin,50) else 0 

    return Pfin + Pth + Pgap -1000*delT -100*delP

weights = [90,10]
mutation = [0,1]

for t in range(iterations ):

    scored_cases = [ ]

    for case in cases:
    
        case_cfd = pd.DataFrame([{
        "g_fin": case[0],
        "t_fin": case[1],
        "N_fin": Nfin(case[0],case[1],50)
        }])

        case_cfd_scaled = X_scaler.transform(case_cfd[input_columns].values) 

        prediction_scaled = S_CFD.predict(case_cfd_scaled)
        prediction = y_scaler.inverse_transform(prediction_scaled)

        J= J_func(prediction[0, 1],prediction[0, 0],case[0],case[1])

        scored_cases.append([J,case])

    scored_cases.sort(key=lambda x:x[0]) #takes the J func and sorts each entry of J and case by the J least to great

    scored_cases = scored_cases[:4] #gets rid of bottom 3 

    cases = [item[1] for item in scored_cases] #just grabbing cases only 

    child1= [cases[0][0],cases[1][1]]
    child2= [cases[1][1],cases[2][2]]
    child3= [cases[0][0],cases[2][2]] 

    childs= [child1,child2,child3]
    
    for child in childs:
        g_mut= rng.choices(mutation, weights=weights, k=1)[0] #k=1 means just draw one thing from the bag and have to select [0] because geneartes a list
        t_mut=rng.choices(mutation, weights=weights, k=1)[0]

        if g_mut == 1:
            child[0]=rng.uniform(.2,50)
        if t_mut== 1:
            child[1]=rng.uniform(.1,50)

    cases.append(childs)




NameError: name 'two' is not defined